# EuIEEE9: C++-Lastfluss im Vergleich zum Paper

Dieses Notebook baut und startet **`SP_Ph1_EuIEEE9_PF`** und vergleicht seine
CSV-Ergebnisse mit **Tabelle 1** aus Bisseling, Weber et al.,
*EMT Simulation of an IEEE 9-Bus Benchmark-Model Adapted to European Transmission
Grid Standards with 100% Inverter-Based Resources*, WIW 2024,
[DOI 10.1049/icp.2024.3847](https://doi.org/10.1049/icp.2024.3847).
Die Vergleichswerte sind aus dem lokalen Paper auf **Seite 2** übertragen.

**Ausführen:** Python-Kernel mit NumPy, pandas und Matplotlib auswählen
(lokal: **Python (DPsim)**), anschließend **Run All / Alle ausführen**.
Ein konfigurierter DPsim-CMake-Build mit aktivierten C++-Beispielen wird benötigt.
Die Python-Bindings von DPsim werden für dieses Notebook nicht benötigt.

Verglichen werden die neun Spannungsbeträge und Winkel, P/Q der drei Generatoren,
die drei Lastvorgaben sowie die Gesamtleistungsbilanz. Tabellen und Diagramme werden
zusätzlich als CSV, PNG und SVG im Ergebnisordner gespeichert.

**Einordnung:** Das Beispiel verwendet statische VD-/PV-Einspeisungen, konstante
PQ-Lasten und π-Leitungen. Das Paper verwendet für die transienten Untersuchungen
Umrichter, konstante Impedanzlasten und frequenzabhängige Leitungen.
Die numerischen Leitungsbeläge fehlen im Paper; im C++-Beispiel stehen dafür
vorläufige PyPSA-Standardwerte. Die Diagramme zeigen daher die Abweichung unseres
gewählten Modells von den veröffentlichten Betriebspunkten.

## 1. Pfade und Ausführung

Das Repository wird vom aktuellen Verzeichnis aus gesucht. Bei einem anderen
Arbeitsverzeichnis `REPO_OVERRIDE` setzen; `BUILD_OVERRIDE` erlaubt einen anderen
bereits konfigurierten Build. Jeder Lauf erhält einen eigenen Ordner, damit keine
CSV eines früheren Laufs versehentlich ausgewertet wird.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import re
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

REPO_OVERRIDE = None  # z. B. Path("/home/ssc/dpsim")
BUILD_OVERRIDE = None  # None -> REPO / "build"; sonst absolut oder relativ zu REPO
BUILD_JOBS = 4
EXAMPLE = "SP_Ph1_EuIEEE9_PF"


def find_repo():
    if REPO_OVERRIDE is not None:
        candidates = [Path(REPO_OVERRIDE).expanduser().resolve()]
    else:
        cwd = Path.cwd().resolve()
        candidates = [
            p for parent in (cwd, *cwd.parents) for p in (parent, parent / "dpsim")
        ]
    for candidate in candidates:
        if (candidate / "dpsim/examples/cxx/Circuits" / f"{EXAMPLE}.cpp").is_file():
            return candidate
    raise FileNotFoundError("DPsim-Repository nicht gefunden. REPO_OVERRIDE setzen.")


REPO = find_repo()
BUILD = Path(BUILD_OVERRIDE).expanduser() if BUILD_OVERRIDE else REPO / "build"
BUILD = (REPO / BUILD).resolve() if not BUILD.is_absolute() else BUILD.resolve()
SOURCE = REPO / "dpsim/examples/cxx/Circuits" / f"{EXAMPLE}.cpp"
EXECUTABLE = BUILD / "dpsim/examples/cxx" / EXAMPLE
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_DIR = REPO / "logs" / "EuIEEE9_Papervergleich" / RUN_ID
REPORT_DIR = RUN_DIR / "vergleich"
RESULT_DIR = RUN_DIR / "logs" / EXAMPLE
CSV_PATH = RESULT_DIR / f"{EXAMPLE}.csv"
SOLVER_LOG = RESULT_DIR / f"{EXAMPLE}_PF_Solver.log"
REPORT_DIR.mkdir(parents=True, exist_ok=False)

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 10,
    }
)
PAPER_COLOR, DPSIM_COLOR = "#D47C16", "#176B91"
print("Repository:", REPO)
print("Build:", BUILD)
print("Ergebnisordner:", RUN_DIR)

In [ ]:
def run_command(command, cwd, log_path, timeout):
    result = subprocess.run(
        command, cwd=cwd, capture_output=True, text=True, timeout=timeout, check=False
    )
    output = result.stdout + result.stderr
    log_path.write_text(output, encoding="utf-8")
    if result.returncode:
        raise RuntimeError(
            f"Befehl fehlgeschlagen ({result.returncode}): {command}\n"
            f"Log: {log_path}\n{output[-6000:]}"
        )
    return output


if not (BUILD / "CMakeCache.txt").is_file():
    raise FileNotFoundError(
        f"Kein konfigurierter Build unter {BUILD}. Zuerst im Terminal ausführen:\n"
        f"cmake -S {REPO} -B {BUILD} -DDPSIM_BUILD_EXAMPLES=ON"
    )

build_output = run_command(
    [
        "cmake",
        "--build",
        str(BUILD),
        "--target",
        EXAMPLE,
        "--parallel",
        str(BUILD_JOBS),
    ],
    REPO,
    RUN_DIR / "build.log",
    900,
)
if not EXECUTABLE.is_file():
    raise FileNotFoundError(f"Executable nicht gefunden: {EXECUTABLE}")

run_output = run_command([str(EXECUTABLE)], RUN_DIR, RUN_DIR / "stdout.log", 60)
if not CSV_PATH.is_file() or not SOLVER_LOG.is_file():
    raise FileNotFoundError(f"Erwartete Ergebnisdateien fehlen in {RESULT_DIR}")

solver_text = SOLVER_LOG.read_text(encoding="utf-8")
iterations = [int(n) for n in re.findall(r"Converged in (\d+) iterations", solver_text)]
if not iterations or re.search(r"not converged|\[E\]", solver_text, re.IGNORECASE):
    raise RuntimeError(f"Kein durchgehend konvergierter Lastfluss. Siehe {SOLVER_LOG}")

print("Build und Ausführung erfolgreich.")
print("NR-Iterationen je protokolliertem Lastfluss:", iterations)
print("CSV:", CSV_PATH)
print("Hinweis: G-Ersatzwert-Warnungen der PiLine betreffen MNA;")
print(
    "im hier gewählten PFSimulation-Verhalten wird der künstliche Querleitwert weggelassen."
)

## 2. Referenzdaten und Bedeutung des Vergleichs

`U_base_kV` stammt aus Abschnitt 2.3 und Tabelle 2; die übrigen Zahlen stammen aus
**Tabelle 1**. Die Referenz bleibt unverändert, einschließlich **−1,32° an Bus 3**.
Die leicht anders gerundeten Generatorleistungen in Abbildung 2 werden hier nicht
mit Tabelle 1 vermischt. Fehlende P/Q-Werte an Bus 4, 7 und 9 bleiben leer.

Alle Spannungen sind **Leiter-Leiter-RMS-Werte**, alle Leistungen sind
**dreiphasige Gesamtleistungen**. Positive Tabellenleistungen bedeuten Einspeisung,
negative Werte Verbrauch. Die CSV loggt Lasten mit positivem Verbrauchsvorzeichen.
Alle Winkel werden auf Bus 1 bezogen; Winkeldifferenzen liegen in [−180°, 180°).

| Größe im Beispiel | Vorgegeben | Unabhängig berechnete Vergleichsgröße |
|---|---|---|
| Bus 1, VD/Slack | Spannungsbetrag und Winkelreferenz | P und Q |
| Bus 2/3, PV | P und Spannungsbetrag | Q und Winkel |
| Bus 4–9, PQ | P und Q; an Verbindungsbussen Nullinjektion | Spannungsbetrag und Winkel |
| Lasten 5/6/8 | Konstante P/Q-Sollwerte | Hier nur Vergleich der eingestellten Parameter |

Übereinstimmende Sollwerte sind eine Kontrolle des Aufbaus. Die Fehlerkennzahlen
unten verwenden deshalb nur die jeweils **berechneten** Größen. Das Paper gibt
keine numerischen Akzeptanzgrenzen für diesen abweichenden Modellfall vor;
aus den Abweichungen wird kein automatisches „bestanden/nicht bestanden“ abgeleitet.

In [ ]:
paper = pd.DataFrame(
    [
        (1, 21, 1.080, 0.00, 361.0, 207.0),
        (2, 27, 1.075, 2.87, 908.0, 363.5),
        (3, 21, 1.049, -1.32, 471.0, 153.5),
        (4, 400, 1.021, -36.23, np.nan, np.nan),
        (5, 400, 1.016, -36.55, -687.5, -275.0),
        (6, 400, 1.017, -36.74, -495.0, -165.0),
        (7, 400, 1.027, -34.99, np.nan, np.nan),
        (8, 400, 1.019, -36.24, -550.0, -192.5),
        (9, 400, 1.026, -35.51, np.nan, np.nan),
    ],
    columns=[
        "Bus",
        "U_base_kV",
        "U_pu_Paper",
        "Winkel_deg_Paper",
        "P_MW_Paper",
        "Q_Mvar_Paper",
    ],
).set_index("Bus")
paper["Typ_im_Beispiel"] = ["VD", "PV", "PV"] + ["PQ"] * 6
display(paper.style.format(precision=3, na_rep="—"))

model_notes = pd.DataFrame(
    [
        ("Frequenz / Spannungsebenen", "50 Hz / 400, 27, 21 kV", "aus dem Paper"),
        ("Leitungslängen", "25, 30, 35, 40, 50, 50 km", "Abbildung 2"),
        ("Trafos", "600 / 1200 / 600 MVA, X = 0,20 pu, YNd11", "Tabelle 2"),
        (
            "Trafo-R",
            "0,015 pu auf eigener Bemessungsbasis",
            "aus Kupferverlusten interpretiert",
        ),
        (
            "Leitungsbeläge",
            "R′ = 0,03 Ω/km; X′ = 0,246 Ω/km; C′ = 13,8 nF/km",
            "vorläufiger PyPSA-Ersatz, keine Paperwerte",
        ),
        ("Leckleitwert", "G′ = 0", "eigene Annahme"),
        ("Lastmodell", "PQ konstant", "eigene Modellwahl; Paper: konstante Impedanz"),
    ],
    columns=["Parameter", "Modellstand des C++-Beispiels", "Herkunft"],
)
display(model_notes)

## 3. C++-Ergebnisse einlesen

Die CSV enthält den stationären Zustand bei 0 s und 1 s. Das sind keine
EMT-Zeitverläufe; ausgewertet wird die letzte Zeile. Das Notebook prüft Spalten,
endliche Zahlen und positive Spannungsbeträge. Für pu werden die oben genannten
Spannungsbasen verwendet; bei Änderung der Spannungsebenen müssen diese mit
angepasst werden.

In [ ]:
raw = pd.read_csv(CSV_PATH, skipinitialspace=True)
raw.columns = raw.columns.str.strip()
voltage_columns = [f"v_bus{bus}.{part}" for bus in paper.index for part in ("re", "im")]
power_columns = [f"Gen{bus}_{pq}" for bus in (1, 2, 3) for pq in ("P", "Q")]
power_columns += [f"Load{bus}_{pq}" for bus in (5, 6, 8) for pq in ("P", "Q")]
required_columns = ["time", *voltage_columns, *power_columns]
missing = sorted(set(required_columns) - set(raw.columns))
if missing:
    raise ValueError(f"CSV-Spalten fehlen: {missing}. Aktuelles C++-Beispiel bauen.")
numeric = raw[required_columns].apply(pd.to_numeric, errors="raise")
if numeric.empty or not np.isfinite(numeric.to_numpy()).all():
    raise ValueError("CSV enthält keine vollständigen, endlichen Ergebniswerte.")
if not numeric["time"].is_monotonic_increasing:
    raise ValueError("Zeitspalte ist nicht monoton steigend.")
last = numeric.iloc[-1]

voltage = pd.Series(
    {
        bus: complex(last[f"v_bus{bus}.re"], last[f"v_bus{bus}.im"])
        for bus in paper.index
    }
)
if (np.abs(voltage) <= 0).any():
    raise ValueError("Mindestens ein Bus hat keine positive Spannungsamplitude.")


def wrap_deg(value):
    return (value + 180.0) % 360.0 - 180.0


bus_compare = paper[
    ["Typ_im_Beispiel", "U_base_kV", "U_pu_Paper", "Winkel_deg_Paper"]
].copy()
bus_compare["U_pu_DPsim"] = np.abs(voltage) / (1000 * paper["U_base_kV"])
bus_compare["U_kV_DPsim"] = np.abs(voltage) / 1000
angles = pd.Series(np.angle(voltage.to_numpy(), deg=True), index=paper.index)
bus_compare["Winkel_deg_DPsim"] = wrap_deg(angles - angles.loc[1])
bus_compare["Winkel_deg_Paper"] = wrap_deg(
    paper["Winkel_deg_Paper"] - paper.loc[1, "Winkel_deg_Paper"]
)
bus_compare["Delta_U_pu"] = bus_compare["U_pu_DPsim"] - bus_compare["U_pu_Paper"]
bus_compare["Delta_U_prozent"] = (
    100 * bus_compare["Delta_U_pu"] / bus_compare["U_pu_Paper"]
)
bus_compare["Delta_Winkel_deg"] = wrap_deg(
    bus_compare["Winkel_deg_DPsim"] - bus_compare["Winkel_deg_Paper"]
)
bus_compare["U_ist_Vorgabe"] = bus_compare.index.isin([1, 2, 3])
bus_compare["Winkel_ist_Vorgabe"] = bus_compare.index == 1

print(f"{len(raw)} CSV-Zeilen; ausgewertet: t = {last['time']:g} s.")
display(
    bus_compare[
        [
            "Typ_im_Beispiel",
            "U_pu_Paper",
            "U_pu_DPsim",
            "Delta_U_pu",
            "Delta_U_prozent",
            "Winkel_deg_Paper",
            "Winkel_deg_DPsim",
            "Delta_Winkel_deg",
        ]
    ].style.format(
        {c: "{:.5f}" for c in bus_compare.select_dtypes(include="number").columns}
    )
)

## 4. Busspannungen und Winkel

Die Winkel an den Generatorbussen und im 400-kV-Netz erhalten getrennte Achsen,
damit kleine Abweichungen neben der 30°-Trafoverschiebung sichtbar bleiben.
Bei den Differenzbalken bedeutet **positiv: DPsim größer als Paper**.

In [ ]:
figures = {}


def finish_figure(name, fig):
    fig.tight_layout()
    figures[name] = fig
    for extension in ("png", "svg"):
        fig.savefig(REPORT_DIR / f"{name}.{extension}", dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)


fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
panels = [
    (axes[0], bus_compare, "U_pu", "Spannungsbetrag |U| [pu]", "Alle Busse"),
    (
        axes[1],
        bus_compare.loc[1:3],
        "Winkel_deg",
        "Winkel relativ zu Bus 1 [°]",
        "Generatorbusse",
    ),
    (
        axes[2],
        bus_compare.loc[4:9],
        "Winkel_deg",
        "Winkel relativ zu Bus 1 [°]",
        "400-kV-Netz",
    ),
]
for ax, data, column, ylabel, title in panels:
    ax.plot(
        data.index,
        data[f"{column}_Paper"],
        "o--",
        color=PAPER_COLOR,
        label="Paper, Tabelle 1",
        linewidth=1.4,
    )
    ax.plot(
        data.index,
        data[f"{column}_DPsim"],
        "x-",
        color=DPSIM_COLOR,
        label="DPsim",
        linewidth=1.4,
        markersize=7,
    )
    ax.set(title=title, xlabel="Bus", ylabel=ylabel, xticks=data.index)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=9)
finish_figure("01_busspannungen", fig)

fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True)
for ax, column, fixed_column, ylabel in [
    (axes[0], "Delta_U_pu", "U_ist_Vorgabe", "Δ|U| = DPsim − Paper [pu]"),
    (axes[1], "Delta_Winkel_deg", "Winkel_ist_Vorgabe", "ΔWinkel = DPsim − Paper [°]"),
]:
    for fixed, color, label in [
        (False, DPSIM_COLOR, "Berechnete Größe"),
        (True, "#AAAAAA", "Vorgegebene Größe"),
    ]:
        subset = bus_compare[bus_compare[fixed_column] == fixed]
        ax.bar(subset.index, subset[column], color=color, label=label)
    ax.axhline(0, color="#555555", linewidth=0.8)
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", alpha=0.25)
    ax.legend(loc="best", fontsize=9)
axes[1].set(xlabel="Bus", xticks=paper.index)
finish_figure("02_busabweichungen", fig)

## 5. Generatorleistungen und Lastvorgaben

Generator-P/Q stammen aus der Ergebnis-CSV. Nur P an Gen2/Gen3 ist vorgegeben;
Slack-P sowie alle Generator-Q sind Lastflussergebnisse.
Die Lastspalten enthalten die tatsächlich im C++-Beispiel eingestellten P/Q-Werte,
keine unabhängig gemessene Lastantwort.

In [ ]:
gen_compare = paper.loc[[1, 2, 3], ["P_MW_Paper", "Q_Mvar_Paper"]].copy()
gen_compare.index.name = "Generatorbus"
gen_compare["P_MW_DPsim"] = [last[f"Gen{i}_P"] / 1e6 for i in gen_compare.index]
gen_compare["Q_Mvar_DPsim"] = [last[f"Gen{i}_Q"] / 1e6 for i in gen_compare.index]
gen_compare["P_Rolle"] = ["Ergebnis: Slack", "Vorgabe: PV", "Vorgabe: PV"]
gen_compare["Q_Rolle"] = "Ergebnis"
for symbol, unit in [("P", "MW"), ("Q", "Mvar")]:
    gen_compare[f"Delta_{symbol}_{unit}"] = (
        gen_compare[f"{symbol}_{unit}_DPsim"] - gen_compare[f"{symbol}_{unit}_Paper"]
    )
    gen_compare[f"Delta_{symbol}_prozent"] = (
        100
        * gen_compare[f"Delta_{symbol}_{unit}"]
        / gen_compare[f"{symbol}_{unit}_Paper"].abs()
    )
display(gen_compare.style.format(precision=4))

load_compare = -paper.loc[[5, 6, 8], ["P_MW_Paper", "Q_Mvar_Paper"]].copy()
load_compare.index.name = "Lastbus"
load_compare["P_MW_CPP_Vorgabe"] = [
    last[f"Load{i}_P"] / 1e6 for i in load_compare.index
]
load_compare["Q_Mvar_CPP_Vorgabe"] = [
    last[f"Load{i}_Q"] / 1e6 for i in load_compare.index
]
load_compare["Delta_P_MW"] = (
    load_compare["P_MW_CPP_Vorgabe"] - load_compare["P_MW_Paper"]
)
load_compare["Delta_Q_Mvar"] = (
    load_compare["Q_Mvar_CPP_Vorgabe"] - load_compare["Q_Mvar_Paper"]
)
display(Markdown("**Lastvorgaben – positives Vorzeichen bedeutet Verbrauch:**"))
display(load_compare.style.format(precision=4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.7))
x = np.arange(3)
for ax, symbol, unit in [(axes[0], "P", "MW"), (axes[1], "Q", "Mvar")]:
    ax.bar(
        x - 0.18,
        gen_compare[f"{symbol}_{unit}_Paper"],
        width=0.36,
        color=PAPER_COLOR,
        label="Paper, Tabelle 1",
    )
    bars = ax.bar(
        x + 0.18,
        gen_compare[f"{symbol}_{unit}_DPsim"],
        width=0.36,
        color=DPSIM_COLOR,
        label="DPsim",
    )
    if symbol == "P":
        for bar in list(bars)[1:]:
            bar.set_hatch("//")
    ax.set(
        xticks=x,
        xticklabels=["Gen1 (Slack)", "Gen2 (PV)", "Gen3 (PV)"],
        ylabel=f"{symbol} [{unit}]",
        title=f"Generator-{symbol}",
    )
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
fig.suptitle("Schraffierte P-Balken: vorgegebene PV-Wirkleistung", fontsize=11)
finish_figure("03_generatorleistungen", fig)

## 6. Fehlerkennzahlen und Bilanz

RMSE ist die Wurzel aus dem Mittel der quadrierten Abweichungen; MAE ist die
mittlere absolute Abweichung. Spannungsfehler werden nur für Bus 4–9,
Winkelfehler für Bus 2–9 ausgewertet. P-Fehler betreffen nur den Slack,
Q-Fehler alle drei Generatoren.

Die Differenz ΣP_Gen − ΣP_Last ist im C++-Modell die aktive Netzverlustleistung.
Für das Paper wird hier lediglich dieselbe **Summendifferenz aus Tabelle 1**
berechnet. Die 7,5 MW sind kein dort separat ausgewiesener Leitungsverlust.
Die Q-Differenz ist die Netto-Blindleistungsaufnahme des Netzes einschließlich
kapazitiver Erzeugung und darf nicht wie eine Wirkleistungsverlustzahl gelesen werden.

In [ ]:
metric_rows = []
for label, values, unit in [
    ("U, Bus 4–9", bus_compare.loc[4:9, "Delta_U_pu"], "pu"),
    ("Winkel, Bus 2–9", bus_compare.loc[2:9, "Delta_Winkel_deg"], "°"),
    ("P, Slack Gen1", gen_compare.loc[[1], "Delta_P_MW"], "MW"),
    ("Q, Gen1–3", gen_compare["Delta_Q_Mvar"], "Mvar"),
]:
    values = values.to_numpy(dtype=float)
    metric_rows.append(
        (
            label,
            len(values),
            unit,
            np.abs(values).mean(),
            np.sqrt(np.mean(values**2)),
            np.abs(values).max(),
        )
    )
metrics = pd.DataFrame(
    metric_rows,
    columns=["Berechnete Größe", "Anzahl", "Einheit", "MAE", "RMSE", "Max. abs."],
)
display(
    metrics.style.format({"MAE": "{:.6f}", "RMSE": "{:.6f}", "Max. abs.": "{:.6f}"})
)

balance = pd.DataFrame(
    {
        "Paper": [
            gen_compare.P_MW_Paper.sum(),
            load_compare.P_MW_Paper.sum(),
            gen_compare.Q_Mvar_Paper.sum(),
            load_compare.Q_Mvar_Paper.sum(),
        ],
        "DPsim": [
            gen_compare.P_MW_DPsim.sum(),
            load_compare.P_MW_CPP_Vorgabe.sum(),
            gen_compare.Q_Mvar_DPsim.sum(),
            load_compare.Q_Mvar_CPP_Vorgabe.sum(),
        ],
    },
    index=[
        "ΣP Generatoren [MW]",
        "ΣP Lasten [MW]",
        "ΣQ Generatoren [Mvar]",
        "ΣQ Lasten [Mvar]",
    ],
)
balance.loc["ΣP Gen − ΣP Last [MW]"] = (
    balance.loc["ΣP Generatoren [MW]"] - balance.loc["ΣP Lasten [MW]"]
)
balance.loc["ΣQ Gen − ΣQ Last [Mvar]"] = (
    balance.loc["ΣQ Generatoren [Mvar]"] - balance.loc["ΣQ Lasten [Mvar]"]
)
balance["Delta_DPsim_minus_Paper"] = balance.DPsim - balance.Paper
display(balance.style.format(precision=5))

max_u_bus = bus_compare.loc[4:9, "Delta_U_pu"].abs().idxmax()
max_angle_bus = bus_compare.loc[2:9, "Delta_Winkel_deg"].abs().idxmax()
display(
    Markdown(
        f"**Auswertung dieses Laufs:** Größte berechnete Spannungsabweichung an Bus {max_u_bus}: "
        f"{bus_compare.loc[max_u_bus, 'Delta_U_pu']:+.5f} pu "
        f"({bus_compare.loc[max_u_bus, 'Delta_U_prozent']:+.3f} %). "
        f"Größte Winkelabweichung an Bus {max_angle_bus}: "
        f"{bus_compare.loc[max_angle_bus, 'Delta_Winkel_deg']:+.4f}°. "
        f"Aktive Netzverluste im C++-Modell: {balance.loc['ΣP Gen − ΣP Last [MW]', 'DPsim']:.4f} MW.\n\n"
        "Diese Zahlen beschreiben den Abstand zur Referenz. Für eine Ursachenanalyse sind "
        "die originalen Leitungsparameter, die Zuordnung der Lastwerte zum Betriebspunkt "
        "und das genaue Trafo-Ersatzschaltbild abzugleichen. Die gerundeten Paperwerte "
        "begrenzen außerdem die erreichbare Vergleichsgenauigkeit."
    )
)

## 7. Export und Quellen

Der Export enthält Referenzwerte, Vergleichstabellen, Kennzahlen, Bilanz,
Abbildungen und Metadaten mit Quellcode-/Executable-Hash. So bleiben spätere
Läufe nach einer Parameteränderung voneinander unterscheidbar.

- **Paperdaten:** Bisseling et al., [DOI 10.1049/icp.2024.3847](https://doi.org/10.1049/icp.2024.3847),
  Seite 2, Tabelle 1; Spannungsbasen aus Abschnitt 2.3 / Tabelle 2.
- **Vorläufiger Leitungstyp im C++-Beispiel:**
  [PyPSA v0.30.2, Al/St 240/40 4-bundle 380.0](https://docs.pypsa.org/v0.30.2/user-guide/components.html#line-types).
  Dieser Datensatz gehört nicht zum KIT-Paper.
- **Modell:** `dpsim/examples/cxx/Circuits/SP_Ph1_EuIEEE9_PF.cpp`.
  Der SP-Transformator berücksichtigt den komplexen Übersetzungsfaktor im Lastfluss.

Ein Vergleich dynamischer VBR-Verläufe ist mit diesem statischen Beispiel noch
nicht möglich. Die vorliegenden Paperkurven stammen außerdem aus Umrichterfällen.

In [ ]:
tables = {
    "paper_tabelle1": paper,
    "modellannahmen": model_notes,
    "busvergleich": bus_compare,
    "generatorvergleich": gen_compare,
    "lastvorgaben": load_compare,
    "fehlerkennzahlen": metrics,
    "leistungsbilanz": balance,
}
for name, frame in tables.items():
    frame.to_csv(REPORT_DIR / f"{name}.csv", encoding="utf-8", float_format="%.10g")


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


metadata = {
    "run_id_utc": RUN_ID,
    "example": EXAMPLE,
    "source": str(SOURCE),
    "source_sha256": sha256(SOURCE),
    "executable": str(EXECUTABLE),
    "executable_sha256": sha256(EXECUTABLE),
    "csv": str(CSV_PATH),
    "csv_sha256": sha256(CSV_PATH),
    "solver_log": str(SOLVER_LOG),
    "converged_iterations": iterations,
    "csv_rows": len(raw),
    "selected_time_s": float(last["time"]),
    "reference": "Bisseling et al., DOI 10.1049/icp.2024.3847, Table 1, page 2",
    "voltage_bases": "Section 2.3 and Table 2; line-to-line RMS",
    "difference_convention": "DPsim minus paper; angles relative to Bus1, wrapped to [-180, 180)",
    "metrics_exclude_prescribed_values": True,
    "model_notes": model_notes.to_dict(orient="records"),
    "python_packages": {"numpy": np.__version__, "pandas": pd.__version__},
}
(REPORT_DIR / "metadaten.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Export abgeschlossen:", REPORT_DIR)
for file in sorted(REPORT_DIR.iterdir()):
    print(" ", file.name)